In [1]:
import pandas as pd
import numpy as np
import yfinance as yf

# Funções uteis

In [2]:
# Obtem historico de negociacao
def obtemHistorico(ticker, data_inicial, data_final):
    histotico= yf.download(ticker, start=data_inicial, end=data_final)
    colunas= []
    for col in histotico.columns:
        colunas.append(col[0])

    return histotico, colunas

In [3]:
# Calcula retorno diario
def retornoDiario(precos):
    retornos= precos.pct_change()
    retornos= retornos.dropna()
    return retornos

In [4]:
# Calcula voltatilidade ultimos n dias
def volatilidadeUltimosNDias(retornos_diarios, n):
    retornos_recentes= retornos_diarios.tail(n)
    return np.std(retornos_recentes) * np.sqrt(252)

# Carregando os dados

In [5]:
# Carrego o histórico de negociação
ticker= "itub4.sa"
data_inicial, data_final= "2015-01-01", "2025-10-20"

df_ativo, colunas= obtemHistorico(ticker, data_inicial, data_final)
df_ativo.columns= colunas
#df_ativo.reset_index(drop=True, inplace=True)

/tmp/ipython-input-3796483879.py:3: FutureWarning: YF.download() has changed argument auto_adjust default to True
  histotico= yf.download(ticker, start=data_inicial, end=data_final)
[*********************100%***********************]  1 of 1 completed


In [6]:
df_ativo.head(3)

,Close,High,Low,Open,Volume
Date,,,,,
2015-01-02,9.377201,9.648844,9.291274,9.493620,22832373
2015-01-05,9.424320,9.521335,9.158221,9.230290,25885620
2015-01-06,9.576766,9.576766,9.321754,9.424314,30498134


In [7]:
df_ativo.tail(3)

,Close,High,Low,Open,Volume
Date,,,,,
2025-10-15,37.450001,37.700001,37.060001,37.320000,21332200
2025-10-16,37.360001,37.799999,37.099998,37.320000,15386000
2025-10-17,37.490002,37.619999,37.009998,37.150002,16960500


In [8]:
df_ativo.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2688 entries, 2015-01-02 to 2025-10-17
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Close   2688 non-null   float64
 1   High    2688 non-null   float64
 2   Low     2688 non-null   float64
 3   Open    2688 non-null   float64
 4   Volume  2688 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 126.0 KB


In [9]:
df_ativo.describe().T

,count,mean,std,min,25%,50%,75%,max
Close,2688.0,1.955369e+01,6.733681e+00,7.395730,1.538927e+01,1.938327e+01,2.293688e+01,3.905235e+01
High,2688.0,1.978701e+01,6.774751e+00,7.443818,1.570332e+01,1.962442e+01,2.320828e+01,3.948216e+01
Low,2688.0,1.932463e+01,6.690685e+00,7.312381,1.519657e+01,1.913267e+01,2.265944e+01,3.884245e+01
Open,2688.0,1.955636e+01,6.730176e+00,7.373288,1.544619e+01,1.939708e+01,2.294026e+01,3.938220e+01
Volume,2688.0,2.943164e+07,1.583291e+07,0.000000,1.874228e+07,2.579607e+07,3.649115e+07,1.767369e+08


# Criando novas features

In [10]:
# Calculando retornos diários
df_ativo['Retorno_Diario']= retornoDiario(df_ativo['Close'])
df_ativo.head(5)

,Close,High,Low,Open,Volume,Retorno_Diario
Date,,,,,,
2015-01-02,9.377201,9.648844,9.291274,9.493620,22832373,NaN
2015-01-05,9.424320,9.521335,9.158221,9.230290,25885620,0.005025
2015-01-06,9.576766,9.576766,9.321754,9.424314,30498134,0.016176
2015-01-07,9.923261,10.072941,9.701512,9.701512,25962086,0.036181
2015-01-08,10.078481,10.117287,9.762489,9.978694,23543326,0.015642


In [11]:
# Calculando volatilidade dos últimos 5 dias
df_ativo['Volatilidade_5_Dias']= df_ativo['Retorno_Diario'].rolling(window=5).apply(lambda x: volatilidadeUltimosNDias(x, 5))
df_ativo.head(10)

,Close,High,Low,Open,Volume,Retorno_Diario,Volatilidade_5_Dias
Date,,,,,,,
2015-01-02,9.377201,9.648844,9.291274,9.493620,22832373,NaN,NaN
2015-01-05,9.424320,9.521335,9.158221,9.230290,25885620,0.005025,NaN
2015-01-06,9.576766,9.576766,9.321754,9.424314,30498134,0.016176,NaN
2015-01-07,9.923261,10.072941,9.701512,9.701512,25962086,0.036181,NaN
2015-01-08,10.078481,10.117287,9.762489,9.978694,23543326,0.015642,NaN
2015-01-09,9.637752,10.020267,9.610032,9.992549,18770294,-0.043730,0.424849
2015-01-12,9.404920,9.682106,9.391061,9.590635,22394540,-0.024158,0.466143
2015-01-13,9.393827,9.643294,9.393827,9.482528,28128289,-0.001179,0.448524
2015-01-14,9.457582,9.598947,9.285727,9.313446,24772372,0.006787,0.344371


In [12]:
# Calculando volatilidade dos últimos 10 dias
df_ativo['Volatilidade_10_Dias']= df_ativo['Retorno_Diario'].rolling(window=10).apply(lambda x: volatilidadeUltimosNDias(x, 10))
df_ativo.head(15)

,Close,High,Low,Open,Volume,Retorno_Diario,Volatilidade_5_Dias,Volatilidade_10_Dias
Date,,,,,,,,
2015-01-02,9.377201,9.648844,9.291274,9.493620,22832373,NaN,NaN,NaN
2015-01-05,9.424320,9.521335,9.158221,9.230290,25885620,0.005025,NaN,NaN
2015-01-06,9.576766,9.576766,9.321754,9.424314,30498134,0.016176,NaN,NaN
2015-01-07,9.923261,10.072941,9.701512,9.701512,25962086,0.036181,NaN,NaN
2015-01-08,10.078481,10.117287,9.762489,9.978694,23543326,0.015642,NaN,NaN
2015-01-09,9.637752,10.020267,9.610032,9.992549,18770294,-0.043730,0.424849,NaN
2015-01-12,9.404920,9.682106,9.391061,9.590635,22394540,-0.024158,0.466143,NaN
2015-01-13,9.393827,9.643294,9.393827,9.482528,28128289,-0.001179,0.448524,NaN
2015-01-14,9.457582,9.598947,9.285727,9.313446,24772372,0.006787,0.344371,NaN


In [13]:
# Calculando volatilidade dos últimos 15 dias
df_ativo['Volatilidade_15_Dias']= df_ativo['Retorno_Diario'].rolling(window=15).apply(lambda x: volatilidadeUltimosNDias(x, 15))
df_ativo.head(20)

,Close,High,Low,Open,Volume,Retorno_Diario,Volatilidade_5_Dias,Volatilidade_10_Dias,Volatilidade_15_Dias
Date,,,,,,,,,
2015-01-02,9.377201,9.648844,9.291274,9.493620,22832373,NaN,NaN,NaN,NaN
2015-01-05,9.424320,9.521335,9.158221,9.230290,25885620,0.005025,NaN,NaN,NaN
2015-01-06,9.576766,9.576766,9.321754,9.424314,30498134,0.016176,NaN,NaN,NaN
2015-01-07,9.923261,10.072941,9.701512,9.701512,25962086,0.036181,NaN,NaN,NaN
2015-01-08,10.078481,10.117287,9.762489,9.978694,23543326,0.015642,NaN,NaN,NaN
2015-01-09,9.637752,10.020267,9.610032,9.992549,18770294,-0.043730,0.424849,NaN,NaN
2015-01-12,9.404920,9.682106,9.391061,9.590635,22394540,-0.024158,0.466143,NaN,NaN
2015-01-13,9.393827,9.643294,9.393827,9.482528,28128289,-0.001179,0.448524,NaN,NaN
2015-01-14,9.457582,9.598947,9.285727,9.313446,24772372,0.006787,0.344371,NaN,NaN
